In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Coding Folders\hilbit_ai_assignment\data\repayment_history.csv")
df.head()

,application_id,emi_number,due_date,emi_amount,amount_paid,paid_date
0,SHF2400000,1,2024-02-09,2642.0,2642.0,2024-02-09
1,SHF2400000,2,2024-03-09,2642.0,2642.0,2024-03-15
2,SHF2400000,3,2024-04-09,2642.0,2642.0,2024-04-09
3,SHF2400000,4,2024-05-09,2642.0,2642.0,2024-05-18
4,SHF2400000,5,2024-06-09,2642.0,2642.0,2024-06-09


In [7]:
emis_per_applicant = df.groupby("application_id")["emi_number"].nunique()
(emis_per_applicant <= 12).sum()
print(f"Number of applicants with 12 or fewer EMIs: {(emis_per_applicant <= 12).sum()}")

Number of applicants with 12 or fewer EMIs: 16423


Main Function below

In [ ]:
def build_target(repayments: pd.DataFrame, as_of: str) -> pd.DataFrame:
    
    # Filter the DataFrame to include only records with emi_number <= 12
    df = repayments[repayments['emi_number'] <= 12].copy()

    df["due_date"] = pd.to_datetime(df["due_date"])
    df["paid_date"] = pd.to_datetime(df["paid_date"])

    as_of_date = pd.to_datetime(as_of)

    #calaculate day difference and unpaid emis 
    paid_dpd = (df["paid_date"] - df["due_date"]).dt.days #dt is a pandas datetime accessor, lets you access whatever info you need from a date.
    unpaid_dpd = (as_of_date - df["due_date"]).dt.days

    # Assign dpd based on whether the emi is paid or unpaid
    df["dpd"] = paid_dpd.where(df["paid_date"].notna(), unpaid_dpd) #df["dpd"] will take paid date if its paid else it will take the unpaid_dpd value
    df["dpd"] = df["dpd"].clip(lower=0)

    df = df[df["due_date"] <= as_of_date]

    bad_flag = (
        df.groupby("application_id")["dpd"]
        .max()
        .ge(90) #Greater than or equal to 90 days past due
        .astype(int)
        .rename("default_90dpd_12m")
    )

    observed_flag = (
        df.groupby("application_id")["emi_number"]
        .count()
        .eq(12) # equals to 12 emis each applicant
        .rename("is_fully_observed")
    )

    result = pd.concat([bad_flag, observed_flag], axis=1).reset_index()

    return result

In [9]:
df["due_date"] = pd.to_datetime(df["due_date"])
df["paid_date"] = pd.to_datetime(df["paid_date"])


    # Calculate day difference and filter
filtered_df = df[(df['paid_date'] - df['due_date']).dt.days > 90]
filtered_df

,application_id,emi_number,due_date,emi_amount,amount_paid,paid_date
199,SHF2400007,11,2024-12-10,8654.0,8654.0,2025-03-28
1441,SHF2400058,15,2025-04-09,3029.0,3029.0,2025-07-18
1689,SHF2400070,4,2024-05-07,3371.0,3371.0,2024-08-07
1918,SHF2400079,9,2024-10-08,2755.0,2755.0,2025-01-29
2939,SHF2400124,2,2024-03-15,2473.0,2473.0,2024-07-18
...,...,...,...,...,...,...
366969,SHF2427579,1,2026-03-28,2828.0,2828.0,2026-07-14
368370,SHF2427852,1,2026-04-10,7172.0,7172.0,2026-07-27
372045,SHF2428587,1,2026-04-26,3189.0,3189.0,2026-08-17
372505,SHF2428679,1,2026-04-26,6566.0,6566.0,2026-08-15


In [2]:
from pathlib import Path
import sys

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "src" / "target.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find src/target.py in the current directory or its parents")

sys.path.insert(0, str(project_root))

from src.target import build_target

In [ ]:
import pandas as pd
from src.target import build_target

apps = pd.read_csv(r"C:\Coding Folders\hilbit_ai_assignment\data\loan_applications.csv")
repayments = pd.read_csv(r"C:\Coding Folders\hilbit_ai_assignment\data\repayment_history.csv")

# Build target
target_df = build_target(repayments, as_of="2026-08-31")

# Merge
df = apps.merge(target_df, on="application_id", how="inner")

# Keep only fully observed loans for training
df_train = df[df["is_fully_observed"] == True].copy()

print(df_train["default_90dpd_12m"].value_counts(normalize=True))
print(df_train.shape)

default_90dpd_12m
0    0.897262
1    0.102738
Name: proportion, dtype: float64
(14863, 34)


In [5]:
df_train["application_date"] = pd.to_datetime(df_train["application_date"], format="%d-%m-%Y")

# Then the split
train = df_train[df_train["application_date"] < "2025-04-01"]
val   = df_train[df_train["application_date"] >= "2025-04-01"]

print("Train size:", train.shape[0], "| Bad rate:", train["default_90dpd_12m"].mean().round(3))
print("Val size:  ", val.shape[0],   "| Bad rate:", val["default_90dpd_12m"].mean().round(3))

Train size: 13875 | Bad rate: 0.106
Val size:   988 | Bad rate: 0.06


In [ ]:
# Drop leakage, identifiers, and post-application columns
DROP_COLS = [
    "application_id", "applicant_id", "application_date",
    "overdue_emis_current", "collection_status",   # leakage
    "gender",                                        # fairness
    "default_90dpd_12m", "is_fully_observed"        # target
]

FEATURES = [c for c in df_train.columns if c not in DROP_COLS]

CAT_COLS = [
    "city_tier", "state", "occupation_type", "income_proof_type",
    "residence_type", "vehicle_segment", "sourcing_channel", "dealer_id"
]

NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]